# **Importação das Bibliotecas**

In [ ]:
import zipfile
import pandas as pd
import re
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import spacy as sp
from bs4 import BeautifulSoup
import string
import random
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Carregamento dos dados**

In [ ]:
path = "/content/drive/MyDrive/nlp-getting-started.zip"
zip_obj = zipfile.ZipFile(path, mode="r")
zip_obj.extractall()
zip_obj.close()

In [ ]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

# **Tratamento e Limpeza dos dados**

In [ ]:
data_train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
target = data_train["target"]
unique, counts = np.unique(target, return_counts=True)
unique, counts

(array([0, 1]), array([4342, 3271]))

In [ ]:
data_train.isna().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [ ]:
data_train.dropna(axis=0, inplace=True)

In [ ]:
data_train.isna().sum()

text      0
target    0
dtype: int64

In [ ]:
data_train.duplicated().sum()

92

In [ ]:
data_train.drop_duplicates(inplace=True)

In [ ]:
data_train.drop(["id", "keyword", "location"], axis=1, inplace=True)

In [ ]:
sns.countplot(data=data_train, x=target);

In [ ]:
target = data_train["target"]
unique, counts = np.unique(target, return_counts=True)
unique, counts

(array([0, 1]), array([4315, 3206]))

# **Pré-processamento**

In [ ]:
sentences = []
labels = []

# Collect sentences and labels into the lists
for _ in data_train:
    sentences.append(data_train['text'])
    labels.append(data_train['target'])

sentences = sentences[0]
labels = labels[0]

In [ ]:
def shuffle_data(sentencas, labels):
    # Combine as sentenças e os rótulos
    combined = list(zip(sentencas, labels))
    # Embaralhar os dados combinados
    random.shuffle(combined)
    # Separar as sentenças e os rótulos embaralhados
    shuffled_sentences, shuffled_labels = zip(*combined)
    return shuffled_sentences, shuffled_labels

In [ ]:
nlp = sp.load("en_core_web_sm")
stop_words = sp.lang.en.STOP_WORDS

In [ ]:
def clean_text(tweet):
    tweet = BeautifulSoup(tweet, "lxml").get_text()
    tweet = re.sub(r"http\S+", "", tweet)
    tweet = re.sub(r"@\w+", "", tweet)
    tweet = re.sub(r"[^a-zA-Z]", " ", tweet).strip()
    tweet = re.sub(r"  +", " ", tweet).strip()
    tweet = tweet.lower()
    doc = nlp(tweet)
    words = [token.text for token in doc if not token.is_stop and token.text not in string.punctuation]
    clean_tweet = " ".join(words)

    return clean_tweet

In [ ]:
sentences = [clean_text(sentence) for sentence in sentences]

<ipython-input-19-70880e77cd88>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  tweet = BeautifulSoup(tweet, "lxml").get_text()


In [ ]:
sentences, labels = shuffle_data(sentences, labels)

In [ ]:
training_sentences, testing_sentences, training_labels, testing_labels = train_test_split(sentences, labels, test_size=0.2, stratify=labels)

In [ ]:
unique, counts = np.unique(training_labels, return_counts=True)
unique, counts

(array([0, 1]), array([3452, 2564]))

In [ ]:
for _ in range(10):
  print(training_sentences[random.randint(0, len(training_sentences) -1 )])

global precipitation measurement satellite captures d image typhoon soudelor
emergency units simulate chemical explosion nu evanston
abortions traumatised times okay decision reasons like rape
refugio oil spill costlier bigger projected
regress slip fall cliff
answer friend isn t blowing wind s yelling survive dust storm
watching man electrocuted roof mumbailocals definitely lesson people learn lessonforlife marinelines mumbai
delmont s onion house purchased plans demolition
s choking hazard nt die
giant cranes holding bridge collapse nearby homes


In [ ]:
for _ in range(10):
  print(testing_sentences[random.randint(0, len(testing_sentences) -1 )])

heat wave adding misery internally displaced gazans
wowo nigerian refugees repatriated cameroon
bomb ass picture mail
m gon na fight taylor soon
hurricane dancers caribbean pirate shipwreck margarita engle henry holt co byr
myanmar displaced rohingya sittwe point return prison like conditions genocide ihhen msf refugees
pic yr old pkk suicide bomber detonated bomb turkey army trench released
firefighters connecticut headed california fight wild fires
cool kids asked wanted hang school panic attack hospital autismawareness
trouble trouble don t way


# **Tokenização e Padding**

In [ ]:
def maior_numero_palavras(sentences):
    # Inicializar variáveis para armazenar a maior contagem de palavras e a sentença correspondente
    max_count = 0
    max_sentence = ""

    # Iterar sobre as sentenças
    for sentence in sentences:
        # Dividir a sentença em palavras
        words = sentence.split()
        # Verificar o número de palavras na sentença atual
        num_words = len(words)
        # Atualizar se o número de palavras atual for maior que o máximo encontrado até agora
        if num_words > max_count:
            max_count = num_words
            max_sentence = sentence

    return max_sentence, max_count

In [ ]:
# Encontrar a sentença com o maior número de palavras
max_sentence, max_len = maior_numero_palavras(training_sentences)

print("Sentença com o maior número de palavras:")
print(f"\n{max_sentence}\n")
print("Número de palavras na sentença:", max_len)

Sentença com o maior número de palavras:

ssp east says car aeg driven young man collided akram s carhot words exchanged n firing air bullet hit tyre

Número de palavras na sentença: 20


In [ ]:
trunc_type='post'
padding_type='post'
oov_token = "<OOV>"
vocab_size = 2**19
embedding_dim = 128

In [ ]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)

# Generate the word index dictionary
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

# Generate and pad the training sequences
training_sequences = tokenizer.texts_to_sequences(training_sentences)
training_padded = pad_sequences(training_sequences, maxlen=max_len, padding=padding_type, truncating=trunc_type)

# Generate and pad the testing sequences
testing_sequences = tokenizer.texts_to_sequences(testing_sentences)
testing_padded = pad_sequences(testing_sequences, maxlen=max_len, padding=padding_type, truncating=trunc_type)

# Convert the labels lists into numpy arrays
training_labels = np.array(training_labels)
testing_labels = np.array(testing_labels)

In [ ]:
for _ in range(10):
  print(training_padded[random.randint(0, len(training_padded) -1 )])

[ 190   37    2 4870    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
[  187 10100  1790  2147   484  1803  1448     2   110   397     0     0
     0     0     0     0     0     0     0     0]
[  10    3 9943 9944  204    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
[6765 6766 2261  630  144  468 1865 1248  178  151 4158   14 6767    0
    0    0    0    0    0    0]
[  83 1214   97  471 2392  274 1239  444 2393    0    0    0    0    0
    0    0    0    0    0    0]
[  594   477  2768   424  2768    28 13215   788 13216     0     0     0
     0     0     0     0     0     0     0     0]
[3598 1103 9913  363 2212 2336  960 1005  107 9914 9915  213  117  358
    0    0    0    0    0    0]
[ 641   37  400  561  355 1230 3316    0    0    0    0    0    0    0
    0    0    0    0    0    0]
[3646 1202  799  778  931  488  489 1033 1203    0    0    0    0    0
    0    0    0    0    0    0]
[522   8 138  34 115 259   0   0 

In [ ]:
for _ in range(10):
  print(testing_padded[random.randint(0, len(testing_padded) -1 )])

[1735 1736    2  458  232  806    2  327 2413    0    0    0    0    0
    0    0    0    0    0    0]
[   9 4345   15 2229 3531  172 4635  172    0    0    0    0    0    0
    0    0    0    0    0    0]
[115 411   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0]
[   4 1177   89  522 1191    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
[   6   24 4773  235 1529  135  804 5154 8748    0    0    0    0    0
    0    0    0    0    0    0]
[ 243   81 8329 2023  498 1721    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
[  442    53   157 10318  2966 10319  1273   466   290     0     0     0
     0     0     0     0     0     0     0     0]
[ 968 1803 1102  357 2409 1948 2013 2842    0    0    0    0    0    0
    0    0    0    0    0    0]
[  20   57  156 5330  171 3027  136  910 5331  323    0    0    0    0
    0    0    0    0    0    0]
[  87   88  362   84   21   70   87 1265   24   21    0    0    0    0
  

# **Construção do modelo**

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_len),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Setup the training parameters
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Print the model summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 20, 128)           67108864  
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 128)               16512     
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 67125505 (256.06 MB)
Trainable params: 67125505 (256.06 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
num_epochs = 15

# Train the model
model.fit(training_padded, training_labels, epochs=num_epochs, validation_data=(testing_padded, testing_labels))

Epoch 1/15
188/188 [==============================] - 308s 2s/step - loss: 0.6041 - accuracy: 0.6652 - val_loss: 0.4449 - val_accuracy: 0.8053
Epoch 2/15
188/188 [==============================] - 324s 2s/step - loss: 0.3397 - accuracy: 0.8619 - val_loss: 0.4360 - val_accuracy: 0.8133
Epoch 3/15
188/188 [==============================] - 320s 2s/step - loss: 0.2136 - accuracy: 0.9194 - val_loss: 0.5044 - val_accuracy: 0.8173
Epoch 4/15
188/188 [==============================] - 330s 2s/step - loss: 0.1470 - accuracy: 0.9451 - val_loss: 0.5585 - val_accuracy: 0.8027
Epoch 5/15
188/188 [==============================] - 311s 2s/step - loss: 0.1127 - accuracy: 0.9596 - val_loss: 0.6079 - val_accuracy: 0.7907
Epoch 6/15
188/188 [==============================] - 324s 2s/step - loss: 0.0905 - accuracy: 0.9691 - val_loss: 0.7062 - val_accuracy: 0.7548
Epoch 7/15
188/188 [==============================] - 322s 2s/step - loss: 0.0805 - accuracy: 0.9709 - val_loss: 0.6722 - val_accuracy: 0.8053

In [ ]:
class DCNN(tf.keras.Model):
  def __init__(self, vocab_size:int, emb_dim:int=128, nb_filters:int=50,
               ffn_units:int=512, nb_classes:int=2, dropout_rate:float=0.1,
               training:bool=False, name:str='dcnn'):
    super(DCNN, self).__init__(name=name)

    self.embedding = layers.Embedding(vocab_size, emb_dim)

    # Camadas de convolução do modelo
    self.bigram = layers.Conv1D(
        filters=nb_filters, kernel_size=2, padding='same', activation='relu')

    self.trigram = layers.Conv1D(
        filters=nb_filters, kernel_size=3, padding='same', activation='relu')

    self.fourgram = layers.Conv1D(
        filters=nb_filters, kernel_size=4, padding='same', activation='relu')

    # Camada de pooling
    self.pool = layers.GlobalMaxPool1D()

    # Camada densa
    self.dense_1 = layers.Dense(units=ffn_units, activation='relu')
    self.dropout= layers.Dropout(rate=dropout_rate)

    if nb_classes == 2:
      self.last_dense = layers.Dense(units=1, activation='sigmoid')
    else:
      self.last_dense = layers.Dense(units=nb_classes, activation='softmax')

  def call(self, inputs, training):
    x = self.embedding(inputs)
    x1 = self.bigram(x)
    x1 = self.pool(x1)
    x2 = self.trigram(x)
    x2 = self.pool(x2)
    x3 = self.fourgram(x)
    x3 = self.pool(x3)

    merget = tf.concat([x1, x2, x3], axis=-1) # (batch_size, 3 * nb_classes)
    merget = self.dense_1(merget)
    merget = self.dropout(merget, training)
    output = self.last_dense(merget)

    return output

In [ ]:
emb_dim = 200
nb_filters = 100
ffn_units = 256
batch_size = 32
nb_classes = len(set(training_labels)) # set para retornar apenas valores únicos
dropout_rate=0.2
epochs=5

In [ ]:
Dcnn = DCNN(vocab_size=vocab_size, emb_dim=emb_dim, nb_filters=nb_filters,
            ffn_units=ffn_units, nb_classes=nb_classes, dropout_rate=dropout_rate)

In [ ]:
if nb_classes == 2:
  Dcnn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
else:
  Dcnn.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
check_point_path = "./" # caminho para salvar o treinamento do modelo na raiz do google drive
ckpt = tf.train.Checkpoint(Dcnn=Dcnn)
ckpt_manager = tf.train.CheckpointManager(ckpt, check_point_path, max_to_keep=5)

if ckpt_manager.latest_checkpoint:
  ckpt.restore(ckpt_manager.latest_checkpoint)
  print("Latest check point restored")

In [ ]:
history = Dcnn.fit(training_padded, training_labels, epochs=num_epochs, validation_data=(testing_padded, testing_labels))

ckpt_manager.save()